# Wildlife Re-ID V3 — Inference notebook

V3: MiewID + SIFT/VLAD/Fisher candidate features, stacked rerankers, open-set detectors, cluster consensus.

Loads the artifacts saved by the training notebook and writes `/kaggle/working/submission.csv`. **No training happens here.**

**Inputs (Add Input):**
- the competition data (`projectwild3`)
- the `wildreid-v3` dataset (same code + precomputed V3 feature bundles as training)
- the **output of the training notebook** (contains `artifacts_v3/`)

Test features are the test bundles built by `features_v3.build_v3_feature_bundle` — the same function and
stage-1 config that built the training bundles — from the provided MiewID descriptors and SIFT keypoints.

## 1. Locate inputs and set up the working layout

In [ ]:
import glob, os, shutil, sys, time, runpy, platform

INPUT = os.environ.get("WILDREID_INPUT", "/kaggle/input")      # Kaggle mounts datasets / notebook outputs here
WORK = os.environ.get("WILDREID_WORK", "/kaggle/working")      # saved as this notebook's output
RUN = os.environ.get("WILDREID_RUN", "/tmp/wildreid_run")      # scratch layout the scripts run in (not saved)
SPECIES = os.environ.get("WILDREID_SPECIES", "hyena leopard").split()
VERSION = "v3"
ART_NAME = f"artifacts_{VERSION}"

def find_one(pattern, what):
    hits = sorted(glob.glob(os.path.join(INPUT, "**", pattern), recursive=True))
    if not hits:
        raise FileNotFoundError(f"Could not find {what} ({pattern}) under {INPUT}. Is it attached via 'Add Input'?")
    return os.path.dirname(hits[0])

def find_code_dataset():
    # V3 and V4 datasets share the code layout; V4 is the one with the exhaustive-SIFT leopard bundle.
    hits = sorted(glob.glob(os.path.join(INPUT, "**", "src_v3", "stage2_v3_models.py"), recursive=True))
    roots = [os.path.dirname(os.path.dirname(h)) for h in hits]
    roots = [r for r in roots if os.path.exists(os.path.join(r, "cache_v3", "v3_bundles_leopard_exh.pkl")) == (VERSION == "v4")]
    if not roots:
        raise FileNotFoundError(f"Could not find the wildreid-{VERSION} code dataset under {INPUT}. Is it attached via 'Add Input'?")
    return roots[0]

CODE_SRC = find_code_dataset()
DATA_SRC = find_one("test.csv", "the competition data")
print("code+cache dataset :", CODE_SRC)
print("competition data   :", DATA_SRC)

# The scripts use repo-relative paths (src_v2/, cache_v3/, reid_student_data_main/), so recreate that layout.
shutil.rmtree(RUN, ignore_errors=True)
os.makedirs(os.path.join(RUN, "cache_v3"))
for d in ("src_v2", "src_v3"):
    shutil.copytree(os.path.join(CODE_SRC, d), os.path.join(RUN, d))
for f in os.listdir(os.path.join(CODE_SRC, "cache_v3")):
    os.symlink(os.path.join(CODE_SRC, "cache_v3", f), os.path.join(RUN, "cache_v3", f))
os.symlink(DATA_SRC, os.path.join(RUN, "reid_student_data_main"))
os.chdir(RUN)
sys.path[:0] = [os.path.join(RUN, "src_v3"), os.path.join(RUN, "src_v2")]

# The cached feature bundles were pickled under NumPy 2 (module path numpy._core). Let NumPy 1.x read them too.
import numpy as np
if int(np.__version__.split(".")[0]) < 2:
    import numpy.core, numpy.core.multiarray
    sys.modules.setdefault("numpy._core", numpy.core)
    sys.modules.setdefault("numpy._core.multiarray", numpy.core.multiarray)

import sklearn, lightgbm, xgboost, catboost, scipy, pandas
print("python", platform.python_version(), "| numpy", np.__version__, "| pandas", pandas.__version__,
      "| scipy", scipy.__version__, "| sklearn", sklearn.__version__, "| lightgbm", lightgbm.__version__,
      "| xgboost", xgboost.__version__, "| catboost", catboost.__version__, "| cpus", os.cpu_count())

ART = find_one(os.path.join(ART_NAME, "v3_params.json"), "the training notebook's " + ART_NAME + " output")
print("artifacts          :", ART)

## 2. Predict (timed)

Code: `src_v3/infer_v3.py` → `src_v3/stage2_v3_models.py` (shared with the training notebook).

In [ ]:
OUT = os.path.join(WORK, "submission.csv")
t0 = time.time()
sys.argv = ["infer_v3.py", "--artifacts", ART, "--test-bundle", "cached", "--out", OUT, "--species", *SPECIES]
runpy.run_path(os.path.join(RUN, "src_v3", "infer_v3.py"), run_name="__main__")
print(f"inference wall time: {time.time() - t0:.1f} s")

## 3. Check the submission file

In [ ]:
import pandas as pd
sub = pd.read_csv(OUT)
test = pd.read_csv(os.path.join(DATA_SRC, "test.csv"))
print(sub.shape, "rows expected:", len(test))
print("all rows predicted:", sub["predictions"].notna().all())
print("new_individual ranked first: %.1f%%" % (100 * sub["predictions"].str.split().str[0].eq("new_individual").mean()))
sub.head(10)